In [0]:
# Load the saved models and test data
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

model_folder = "/Volumes/workspace/mortgage_risk/model_artifacts"

gbt_model = PipelineModel.load(
    model_folder + "/gradient_boosted_trees_v1"
)

calibration_model = PipelineModel.load(
    model_folder + "/gbt_v1_calibration_v1"
)

test = spark.table("workspace.mortgage_risk.model_test")

score_table = "workspace.mortgage_risk.test_scores_gbt_v1"

In [0]:
# Check the test period
display(
    test.agg(
        F.count("*").alias("rows"),
        F.sum("label_12m").alias("events"),
        F.min("prediction_month").alias("first_month"),
        F.max("prediction_month").alias("last_month")
    )
)

rows,events,first_month,last_month
3306965,25184,2024-01-01,2025-03-01


In [0]:
# Generate predictions
predictions = gbt_model.transform(test)

predictions = predictions.withColumn(
    "risk_score",
    vector_to_array("probability")[1]
)

predictions = calibration_model.transform(predictions)

test_scores = predictions.withColumn(
    "calibrated_risk",
    vector_to_array("calibrated_probability")[1]
).select(
    "loan_id",
    "prediction_month",
    "label_12m",
    "delinquency_months",
    "risk_score",
    "calibrated_risk"
)

In [0]:
#Save the predictions
if spark.catalog.tableExists(score_table):
    test_scores = spark.table(score_table)
    print("Loaded the saved test scores.")

else:
    test_scores.write.format("delta").mode("error").saveAsTable(
        score_table
    )

    test_scores = spark.table(score_table)
    print("Test scores saved.")

Test scores saved.


In [0]:
# Check probability accuracy on the test data
test_metrics = test_scores.agg(
    F.count("*").alias("rows"),

    F.avg("label_12m").alias("observed_event_rate"),

    F.avg("risk_score").alias("original_average_risk"),

    F.avg("calibrated_risk").alias("calibrated_average_risk"),

    F.avg(
        F.pow(F.col("risk_score") - F.col("label_12m"), 2)
    ).alias("original_brier"),

    F.avg(
        F.pow(F.col("calibrated_risk") - F.col("label_12m"), 2)
    ).alias("calibrated_brier")
)

display(test_metrics)

rows,observed_event_rate,original_average_risk,calibrated_average_risk,original_brier,calibrated_brier
3306965,0.007615441953573745,0.039614361569118565,0.007796536601001355,0.0077330809212028916,0.006549275282261904


In [0]:
    #### measure how many events your monthly 5% alerts capture ####

In [0]:
from pyspark.sql.window import Window
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Load the saved scores and calculate PR AUC
test_scores = spark.table(
    "workspace.mortgage_risk.test_scores_gbt_v1"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label_12m",
    rawPredictionCol="risk_score",
    metricName="areaUnderPR",
    numBins=10000
)

pr_auc = pr_evaluator.evaluate(test_scores)

print("Test PR AUC:", pr_auc)

Test PR AUC: 0.23931103600016104


In [0]:
#Prepare both populations
alert_scores = test_scores.withColumn(
    "population",
    F.lit("All eligible loans")
)

healthy_scores = test_scores.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population",
    F.lit("Current status 00")
)

alert_scores = alert_scores.unionByName(healthy_scores)

In [0]:
# Rank the observations each month
month_window = Window.partitionBy(
    "population",
    "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

ranked = alert_scores.withColumn(
    "monthly_rows",
    F.count("*").over(month_window)
)

ranked = ranked.withColumn(
    "risk_rank",
    F.row_number().over(risk_window)
)

In [0]:
# Flag the highest-risk 5%
ranked = ranked.withColumn(
    "alert",
    F.when(
        F.col("risk_rank") <= F.ceil(F.col("monthly_rows") * 0.05),
        1
    ).otherwise(0)
)

ranked = ranked.withColumn(
    "event_caught",
    F.when(
        F.col("alert") == 1,
        F.col("label_12m")
    ).otherwise(0)
)

In [0]:
# Calculate precision, recall, and lift
alert_results = ranked.groupBy("population").agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.sum("alert").alias("alerts"),
    F.sum("event_caught").alias("events_caught")
)

alert_results = alert_results.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")
    )
)

alert_results = alert_results.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")
    )
)

alert_results = alert_results.withColumn(
    "lift",
    F.when(
        F.col("events") > 0,
        F.col("precision") / (F.col("events") / F.col("rows"))
    )
)

display(alert_results.orderBy("population"))

population,rows,events,alerts,events_caught,precision,recall,lift
All eligible loans,3306965,25184,165358,13958,0.08441079355096215,0.5542407878017789,11.084162162295804
Current status 00,3275008,16048,163757,5358,0.03271921200315101,0.33387337986041876,6.677198471087712


In [0]:
alert_table = "workspace.mortgage_risk.test_alerts_gbt_v1"

if spark.catalog.tableExists(alert_table):
    print("Test alert results are already saved.")

else:
    alert_results.write.format("delta").mode("error").saveAsTable(
        alert_table
    )

    print("Test alert results saved.")

Test alert results are already saved.
